# Temporal Ruleformer Pipeline

Divides the FinDKG dataset into time groups (default size 10), trains a fresh Ruleformer
model per group, decodes rules, applies them to the group's subgraph, and tags each
prediction with `timestamp = g_end + 1`.

All predictions are accumulated into `sym_triplets.csv` with 4 columns:
`h_id  r_id  t_id  timestamp`

## Cell 1 — GPU check + Mount Google Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Clone repos

In [ ]:
import os

# Clone Ruleformer
if not os.path.exists('/content/ruleformer-findkg'):
    !git clone https://github.com/rud-rax/ruleformer-findkg.git /content/ruleformer-findkg
else:
    print('ruleformer-findkg already cloned')

# Clone FinDKG (for DKG.symbolic helpers)
if not os.path.exists('/content/FinDKG'):
    !git clone https://github.com/rud-rax/FinDKG.git /content/FinDKG
else:
    !git -C /content/FinDKG pull
    print('FinDKG updated')

## Cell 3 — Configure paths

**Edit this cell** to set your Google Drive folder and hyperparameters.

In [ ]:
import sys
sys.path.insert(0, '/content/FinDKG')

# ── Google Drive paths (edit these) ──────────────────────────────────────────
GDRIVE_HOME  = '/content/drive/MyDrive/FinDKG'          # your GDrive folder
DATASET_PATH = os.path.join(GDRIVE_HOME, 'FinDKG_dataset', 'FinDKG')
EXPS_PATH    = os.path.join(GDRIVE_HOME, 'experiments_temporal')
OUTPUT_PATH  = os.path.join(GDRIVE_HOME, 'sym_triplets.csv')

# ── Ruleformer location (no need to change) ───────────────────────────────────
RF_ROOT = '/content/ruleformer-findkg'

# ── Pipeline hyperparameters ──────────────────────────────────────────────────
GROUP_SIZE  = 10    # timestamps per group (groups: 0–9, 10–19, …)
JUMP        = 2     # rule length (hops in subgraph)
PADDING     = 100   # max subgraph size
MAXN        = 40    # max neighbours per relation
EPOCHS      = 10    # training epochs per group
BATCH_SIZE  = 16
N_HEAD      = 6
D_V         = 64
N_LAYERS    = 2
DROPOUT     = 0.1
SAVESTEP    = 5

# ── Rule decoding thresholds ──────────────────────────────────────────────────
THE_REL     = 0.3
THE_REL_MIN = 0.1
THE_ALL     = 0.05
MIN_WEIGHT  = 0.1
MIN_COUNT   = 1

os.makedirs(EXPS_PATH, exist_ok=True)
print(f'Dataset : {DATASET_PATH}')
print(f'EXPS    : {EXPS_PATH}')
print(f'Output  : {OUTPUT_PATH}')

## Cell 4 — Load data + build time groups

In [ ]:
from DKG.symbolic.temporal_pipeline import (
    load_all_triplets, load_id2name, write_global_vocab, make_groups
)

# Load all triplets from train.txt, valid.txt, test.txt
all_triplets = load_all_triplets(DATASET_PATH)
print(f'Total triplets loaded: {len(all_triplets):,}')

# Load global ID ↔ name mappings
id2ent, id2rel = load_id2name(DATASET_PATH)
ent2id = {v: k for k, v in id2ent.items()}
rel2id = {v: k for k, v in id2rel.items()}
print(f'Entities: {len(id2ent):,}  |  Relations: {len(id2rel):,}')

# Write global entities.txt and relations.txt to Ruleformer DATASET dir
write_global_vocab(id2ent, id2rel, RF_ROOT, dataset='FinDKG')

# Build time groups
groups = make_groups(all_triplets, GROUP_SIZE)
print(f'\nTotal groups: {len(groups)}')
for i, (g_start, g_end) in enumerate(groups):
    count = sum(1 for t in all_triplets if g_start <= t[3] <= g_end)
    print(f'  Group {i:3d}: t={g_start:3d}–{g_end:3d}  ({count:,} triplets)')

## Cell 5 — Process all groups (train → decode → apply)

**To process only the first N groups** for a quick test, change `groups` to `groups[:N]`.

Groups that already have a `rules.txt` on Google Drive are skipped automatically.

In [ ]:
from DKG.symbolic.temporal_pipeline import process_group
from tqdm.auto import tqdm

all_predictions = []

# Change groups[:1] to groups[:N] or groups to control how many groups to process
for g_idx, (g_start, g_end) in enumerate(tqdm(groups, desc="All groups")):
    print(f"\n{'='*60}")
    print(f"GROUP {g_idx}: timestamps {g_start}–{g_end}")
    print(f"{'='*60}")

    preds = process_group(
        g_idx=g_idx,
        g_start=g_start,
        g_end=g_end,
        all_triplets=all_triplets,
        groups=groups,
        id2ent=id2ent,
        id2rel=id2rel,
        ent2id=ent2id,
        rel2id=rel2id,
        rf_root=RF_ROOT,
        exps_path=EXPS_PATH,
        group_size=GROUP_SIZE,
        jump=JUMP,
        padding=PADDING,
        maxn=MAXN,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        n_head=N_HEAD,
        d_v=D_V,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
        savestep=SAVESTEP,
        the_rel=THE_REL,
        the_rel_min=THE_REL_MIN,
        the_all=THE_ALL,
        min_weight=MIN_WEIGHT,
        min_count=MIN_COUNT,
        dataset='FinDKG',
        skip_if_exists=True,   # set False to force rerun
    )

    print(f'  → {len(preds):,} predictions for t={g_end+1}')
    all_predictions.extend(preds)

print(f'\nTotal predictions across all groups: {len(all_predictions):,}')

## Cell 6 — Inspect predictions for a specific group

In [ ]:
# Show predictions from a specific group — adjust g_to_inspect
g_to_inspect = 9   # group [90–99] → predictions at t=100
pred_ts = groups[g_to_inspect][1] + 1

subset = [p for p in all_predictions if p[3] == pred_ts]
print(f'Group {g_to_inspect} ({groups[g_to_inspect][0]}–{groups[g_to_inspect][1]}) '
      f'→ predictions at t={pred_ts}: {len(subset):,}')

for h, r, t, ts in subset[:15]:
    h_name = id2ent.get(h, str(h))
    r_name = id2rel.get(r, str(r))
    t_name = id2ent.get(t, str(t))
    print(f'  ({h_name}, {r_name}, {t_name}, t={ts})')

## Cell 7 — Write sym_triplets.csv to Google Drive

In [ ]:
from DKG.symbolic.temporal_pipeline import write_sym_triplets

write_sym_triplets(all_predictions, OUTPUT_PATH)

# Quick summary
import pandas as pd
df = pd.read_csv(OUTPUT_PATH, sep='\t', header=None, names=['h_id','r_id','t_id','timestamp'])
print(df.describe())
print(f'\nTimestamps in output: {sorted(df["timestamp"].unique())}')